# HydroClaude 教程3：水锤分析
# Tutorial 3: Water Hammer Analysis

**作者 / Author**: HydroClaude Development Team  
**日期 / Date**: 2025-10-30  
**难度 / Level**: 高级 / Advanced  
**时长 / Duration**: 35分钟

---

## 📚 教程目标 / Tutorial Objectives

在这个教程中，你将学会：
1. 理解水锤现象的物理机制
2. 使用Joukowsky公式估算压力升高
3. 使用MOC方法进行瞬态模拟
4. 分析阀门关闭引起的水锤
5. 设计水锤防护措施

---

## 🎯 问题描述 / Problem Description

水锤（Water Hammer）是管道中流体突然停止或改变流速时产生的压力波动现象。

**工程案例**：
- 泵站突然停电
- 阀门快速关闭
- 水轮机负荷突变

**危害**：
- 管道爆裂（正压过大）
- 管道瘪塌（负压过大）
- 设备损坏

---

## Step 1: 导入模块

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.getcwd()))

from transient.moc_solver import MOCSolver
from transient.boundary_conditions import (
    ReservoirBC,
    ValveBC,
    CentrifugalPumpBC
)

print("✅ 模块导入成功！")

## Step 2: Joukowsky公式 - 快速估算

###  2.1 理论基础

Joukowsky公式用于估算突然停止时的最大压力升高：

$$
\Delta H = \frac{a \cdot V}{g}
$$

其中：
- ΔH: 水头升高 (m)
- a: 水锤波速 (m/s)
- V: 初始流速 (m/s)
- g: 重力加速度 (9.81 m/s²)

水锤波速：
$$
a = \sqrt{\frac{K/\rho}{1 + \frac{K \cdot D}{E \cdot e}}}
$$

其中：
- K: 水的体积模量 (~2.19 GPa)
- ρ: 水的密度 (1000 kg/m³)
- D: 管道内径 (m)
- E: 管壁弹性模量 (钢管~200 GPa)
- e: 管壁厚度 (m)

In [ ]:
# 工程案例：供水管道突然停泵
Q = 0.2       # 流量 (m³/s)
D = 0.5       # 管径 (m)
L = 1000      # 管长 (m)
e = 0.01      # 管壁厚度 (m)

# 材料属性
K = 2.19e9    # 水的体积模量 (Pa)
rho = 1000    # 水密度 (kg/m³)
E = 200e9     # 钢管弹性模量 (Pa)
g = 9.81      # 重力加速度

# 计算流速
A = np.pi * (D/2)**2
V = Q / A

# 计算波速
a = np.sqrt((K/rho) / (1 + K*D/(E*e)))

# Joukowsky公式
delta_H = a * V / g

print("📊 Joukowsky公式估算:")
print("=" * 60)
print(f"管道参数:")
print(f"  流量 Q = {Q} m³/s")
print(f"  管径 D = {D} m")
print(f"  管长 L = {L} m")
print(f"  壁厚 e = {e} m")
print(f"\n初始状态:")
print(f"  流速 V = {V:.3f} m/s")
print(f"\n水锤参数:")
print(f"  波速 a = {a:.1f} m/s")
print(f"  压力升高 ΔH = {delta_H:.2f} m")
print(f"  压力升高 ΔP = {delta_H*9.81/1000:.3f} MPa")
print(f"\n临界关闭时间:")
T_critical = 2 * L / a
print(f"  T_c = 2L/a = {T_critical:.2f} 秒")
print(f"\n  关闭时间 < {T_critical:.2f}s: 直接水锤（最危险）")
print(f"  关闭时间 > {T_critical:.2f}s: 间接水锤（较安全）")
print("=" * 60)

## Step 3: MOC数值模拟 - 精确分析

### 3.1 方法特征线方法 (Method of Characteristics)

MOC将偏微分方程转化为沿特征线的常微分方程：

**动量方程**（沿C+ 特征线）：
$$
H_P = C_P - B_P \cdot Q_P
$$

**连续性方程**（沿C- 特征线）：
$$
H_P = C_M + B_M \cdot Q_P
$$

其中：
- C_P, C_M: 由上一时刻状态计算的常数
- B_P, B_M: 阻力系数

In [ ]:
# 管道离散化
N = 20  # 节点数
dx = L / (N - 1)

# 时间步长（满足CFL条件）
dt = dx / a

print(f"\n🔧 MOC数值设置:")
print("=" * 60)
print(f"空间离散:")
print(f"  节点数 N = {N}")
print(f"  网格间距 Δx = {dx:.2f} m")
print(f"\n时间离散:")
print(f"  时间步长 Δt = {dt:.4f} s")
print(f"  CFL数 = a*Δt/Δx = {a*dt/dx:.2f} (应=1.0)")
print("=" * 60)

### 3.2 设置边界条件

In [ ]:
# 上游边界：水库（恒定水头）
H0_upstream = 30.0  # 上游水头 (m)

upstream_bc = ReservoirBC(
    name="Upstream Reservoir",
    head=H0_upstream
)

# 下游边界：阀门（快速关闭）
t_close = 5.0  # 关闭时间 (s)

def valve_closure_curve(t):
    """阀门开度随时间变化
    
    Args:
        t: 时间 (s)
    
    Returns:
        tau: 阀门开度 (0-1)
    """
    if t < t_close:
        return 1.0 - t / t_close  # 线性关闭
    else:
        return 0.0  # 完全关闭

downstream_bc = ValveBC(
    name="Downstream Valve",
    closure_curve=valve_closure_curve,
    K_valve=10.0  # 阀门局部损失系数
)

print(f"\n🔧 边界条件设置:")
print("=" * 60)
print(f"上游（x=0）: 水库，H = {H0_upstream} m")
print(f"下游（x=L）: 阀门，关闭时间 = {t_close} s")
print(f"\n关闭类型判断:")
if t_close < T_critical:
    print(f"  {t_close}s < {T_critical:.2f}s → 直接水锤 ⚠️")
else:
    print(f"  {t_close}s > {T_critical:.2f}s → 间接水锤 ✓")
print("=" * 60)

### 3.3 运行MOC模拟

In [ ]:
# 创建MOC求解器
solver = MOCSolver(
    L=L,
    D=D,
    a=a,
    f=0.02,  # 摩阻系数
    N=N,
    dt=dt
)

# 设置边界条件
solver.set_boundary('upstream', upstream_bc)
solver.set_boundary('downstream', downstream_bc)

# 初始条件
H_initial = np.full(N, H0_upstream)  # 初始水头
Q_initial = np.full(N, Q)            # 初始流量

# 模拟时间
t_max = 20.0  # 模拟20秒

print(f"\n🚀 开始MOC模拟...")
print(f"  模拟时长: {t_max} s")
print(f"  时间步数: {int(t_max/dt)}")
print()

# 运行模拟
result = solver.solve(
    H_initial=H_initial,
    Q_initial=Q_initial,
    t_max=t_max,
    save_interval=10  # 每10步保存一次
)

print(f"\n✅ 模拟完成！")
print(f"  保存的时间步: {len(result['t'])}")

## Step 4: 结果分析

### 4.1 下游节点压力历程

In [ ]:
# 提取下游节点（x=L）的压力历程
t_history = result['t']
H_downstream = result['H'][:, -1]  # 最后一个节点（下游）

# 计算压力（相对于初始状态）
pressure_rise = H_downstream - H0_upstream

# 找到最大压力
max_pressure_rise = np.max(pressure_rise)
max_pressure_time = t_history[np.argmax(pressure_rise)]

print(f"\n📊 下游节点压力分析:")
print("=" * 60)
print(f"初始水头: {H0_upstream} m")
print(f"最大水头: {H0_upstream + max_pressure_rise:.2f} m")
print(f"压力升高: {max_pressure_rise:.2f} m (发生在 t={max_pressure_time:.2f}s)")
print(f"\n对比Joukowsky公式:")
print(f"  MOC模拟: ΔH = {max_pressure_rise:.2f} m")
print(f"  Joukowsky: ΔH = {delta_H:.2f} m")
print(f"  误差: {abs(max_pressure_rise - delta_H)/delta_H*100:.1f}%")
print("=" * 60)

### 4.2 可视化结果

In [ ]:
# 创建图表
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# 子图1：下游压力历程
ax1.plot(t_history, H_downstream, 'b-', linewidth=2, label='MOC Simulation')
ax1.axhline(y=H0_upstream, color='g', linestyle='--', label=f'Initial Head ({H0_upstream}m)')
ax1.axhline(y=H0_upstream + delta_H, color='r', linestyle='--', 
            label=f'Joukowsky Estimate ({H0_upstream + delta_H:.1f}m)')
ax1.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Head (m)', fontsize=11, fontweight='bold')
ax1.set_title('Downstream Head History', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# 子图2：下游流量历程
Q_downstream = result['Q'][:, -1]
ax2.plot(t_history, Q_downstream * 1000, 'b-', linewidth=2)
ax2.axhline(y=0, color='r', linestyle='--')
ax2.axvline(x=t_close, color='orange', linestyle='--', label=f'Valve Closure ({t_close}s)')
ax2.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Flow Rate (L/s)', fontsize=11, fontweight='bold')
ax2.set_title('Downstream Flow History', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# 子图3：压力沿程分布（不同时刻）
times_to_plot = [0, int(len(t_history)*0.25), int(len(t_history)*0.5), int(len(t_history)*0.75), -1]
x_nodes = np.linspace(0, L, N)
colors = ['blue', 'green', 'orange', 'red', 'purple']

for i, color in zip(times_to_plot, colors):
    ax3.plot(x_nodes, result['H'][i, :], color=color, linewidth=2, 
             label=f't={t_history[i]:.2f}s', marker='o', markersize=4)

ax3.set_xlabel('Distance (m)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Head (m)', fontsize=11, fontweight='bold')
ax3.set_title('Head Distribution Along Pipe', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 子图4：压力波传播动画（时空图）
T, X = np.meshgrid(t_history, x_nodes)
im = ax4.contourf(T, X, result['H'].T, levels=20, cmap='RdYlBu_r')
plt.colorbar(im, ax=ax4, label='Head (m)')
ax4.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Distance (m)', fontsize=11, fontweight='bold')
ax4.set_title('Pressure Wave Propagation', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ 可视化完成！")

## Step 5: 水锤防护措施

### 5.1 常见防护措施

**1. 延长关闭时间**
- 使用缓闭阀
- 自动控制系统

**2. 设置水锤消除装置**
- 空气罐
- 调压塔
- 单向调压阀

**3. 改善管道系统**
- 降低流速（增大管径）
- 缩短管道长度
- 分段设置

**4. 泵站专用措施**
- 飞轮增加惯性
- 缓冲罐
- 自动排气阀

### 5.2 效果对比

In [ ]:
# 对比不同关闭时间的效果
closure_times = [1.0, 3.0, 5.0, 10.0]  # 不同的关闭时间
max_pressures = []

print(f"\n🔧 关闭时间影响分析:")
print("=" * 70)
print(f"{'关闭时间(s)':<12} {'最大压升(m)':<15} {'相对Joukowsky':<20} {'类型'}")
print("=" * 70)

for t_c in closure_times:
    # 创建新的阀门边界条件
    def valve_curve(t):
        return max(0, 1.0 - t/t_c)
    
    bc = ValveBC("Test Valve", closure_curve=valve_curve, K_valve=10.0)
    solver.set_boundary('downstream', bc)
    
    # 模拟
    result = solver.solve(H_initial, Q_initial, t_max, save_interval=10)
    
    # 提取最大压力
    H_max = np.max(result['H'][:, -1])
    delta_H_sim = H_max - H0_upstream
    max_pressures.append(delta_H_sim)
    
    # 判断类型
    hammer_type = "直接水锤" if t_c < T_critical else "间接水锤"
    
    print(f"{t_c:<12.1f} {delta_H_sim:<15.2f} {delta_H_sim/delta_H*100:<20.1f}% {hammer_type}")

print("=" * 70)
print(f"\n💡 建议: 关闭时间应 > {T_critical:.1f}s 以避免直接水锤")

### 5.3 关闭时间优化曲线

In [ ]:
# 绘制关闭时间-压力升高曲线
plt.figure(figsize=(10, 6))

plt.plot(closure_times, max_pressures, 'bo-', linewidth=2, markersize=8, label='Simulation')
plt.axhline(y=delta_H, color='r', linestyle='--', linewidth=2, label=f'Joukowsky ({delta_H:.1f}m)')
plt.axvline(x=T_critical, color='orange', linestyle='--', linewidth=2, 
            label=f'Critical Time ({T_critical:.1f}s)')

plt.xlabel('Closure Time (s)', fontsize=12, fontweight='bold')
plt.ylabel('Maximum Pressure Rise (m)', fontsize=12, fontweight='bold')
plt.title('Effect of Closure Time on Water Hammer', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)

# 添加注释
plt.annotate('Direct Water Hammer\n(Most Dangerous)', 
             xy=(closure_times[0], max_pressures[0]), 
             xytext=(closure_times[0]+1, max_pressures[0]+2),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, fontweight='bold', color='red')

plt.annotate('Indirect Water Hammer\n(Safer)', 
             xy=(closure_times[-1], max_pressures[-1]), 
             xytext=(closure_times[-1]-2, max_pressures[-1]+2),
             arrowprops=dict(arrowstyle='->', color='green', lw=2),
             fontsize=10, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print("\n✅ 优化分析完成！")

## 🎓 知识要点总结 / Key Takeaways

### 1. 水锤基本特征
- **波速**: 钢管中约1000-1400 m/s
- **周期**: T = 2L/a（压力波往返时间）
- **衰减**: 摩阻导致压力波逐渐衰减

### 2. 分析方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| Joukowsky公式 | 快速、简便 | 不考虑摩阻和边界 | 初步估算 |
| MOC数值模拟 | 精确、全面 | 计算量大 | 详细设计 |

### 3. 设计准则
- **关闭时间**: t_close > 2L/a（临界时间）
- **最大压力**: 不超过管道设计压力的1.5倍
- **负压控制**: 避免产生汽蚀

### 4. 工程实践
- 泵站设计：必须进行水锤计算
- 长距离输水：设置调压设施
- 阀门控制：采用缓闭方式

---

## ⚠️ 安全提示 / Safety Notes

1. **设计阶段**: 必须进行水锤校核
2. **施工阶段**: 严格按设计要求施工
3. **运行阶段**: 禁止快速操作阀门
4. **维护阶段**: 定期检查防护装置

---

## 🚀 下一步学习 / Next Steps

1. **泵站水锤**: 研究泵特性曲线的影响
2. **复杂边界**: 空气罐、调压塔设计
3. **多管系统**: 管网水锤分析

---

**祝学习愉快！/ Happy Learning!** 🎉